# SCRIPT 4 — HYDROLOGIC AND FLOOD METRIC ASSESSMENT
Brahmaputra Morphodynamics Study — Flood Metric Assessment
Bhomoraguri–Pandu Reach






# Section 1: Data Loading and Quality Control
NEW INDICES (Section 4):


*   RIFW  — Reach Internal Flood Work

*   UDR   — Upstream Dominance Ratio
*   IRTC  — Intra-reach Tributary Contribution

*    EHE   — Effective Hydraulic Energy (Wolman-Miller, k = 1.7)

peak_doy_deviation — signed lag from long-term typical flood peak (DOY 196)
  FPI_bhomo / FPI_pandu — Flood Persistence Index (fraction of season above bkf)



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import curve_fit
from scipy.stats import pearsonr
from scipy.signal import correlate
from scipy.ndimage import gaussian_filter1d
import os
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

ROOT           = '/content/drive/MyDrive/BRAHMA_MORPHO'
DISCHARGE_PATH = f'{ROOT}/data/Discharge_2001_2025_Pandu_Bhomoraguri.xlsx'
WL_PATH        = f'{ROOT}/data/Pandu_and_Bhomoraguri_WL_2001_2025.csv'
FIGURES_DIR    = f'{ROOT}/figures/script4'
os.makedirs(FIGURES_DIR, exist_ok=True)

print('Section 1: Data Loading and QC')

In [ ]:
df_raw = pd.read_excel(DISCHARGE_PATH)
df_raw['dataTime'] = pd.to_datetime(df_raw['dataTime'])
df_raw['year']  = df_raw['dataTime'].dt.year
df_raw['month'] = df_raw['dataTime'].dt.month
df_raw['date']  = df_raw['dataTime'].dt.date

pandu_raw = df_raw[df_raw['stationName'] == 'Pandu'].copy()
bhomo_raw = df_raw[df_raw['stationName'] == 'Bhomoraguri'].copy()

print(f'Pandu raw records    : {len(pandu_raw)}')
print(f'Bhomoraguri raw records: {len(bhomo_raw)}')

Physically justified QC thresholds
Based on basin area, tributary network, and published maximum flood records
Bhomoraguri: drains ~400,000 km² upstream — max credible ~55,000 m³/s
Pandu: additional Kopili, Kollong, Kulsi tributaries — max credible ~72,000 m³/s
Zero or negative discharge is physically impossible for a perennial river

In [ ]:

PANDU_MAX_CREDIBLE = 72000
PANDU_MIN_CREDIBLE = 100
BHOMO_MAX_CREDIBLE = 55000
BHOMO_MIN_CREDIBLE = 100

pandu_raw['is_valid'] = pandu_raw['dataValue'].between(PANDU_MIN_CREDIBLE, PANDU_MAX_CREDIBLE)
bhomo_raw['is_valid'] = bhomo_raw['dataValue'].between(BHOMO_MIN_CREDIBLE, BHOMO_MAX_CREDIBLE)

print('\n=== QC Summary ===')
print(f'Pandu   — flagged: {(~pandu_raw["is_valid"]).sum()} of {len(pandu_raw)} records')
print(f'Bhomo   — flagged: {(~bhomo_raw["is_valid"]).sum()} of {len(bhomo_raw)} records')

print('\nPandu flagged values:')
print(pandu_raw[~pandu_raw['is_valid']][['dataTime', 'dataValue']].to_string())
print('\nBhomoraguri flagged values:')
print(bhomo_raw[~bhomo_raw['is_valid']][['dataTime', 'dataValue']].to_string())

# Apply filter — keep only valid records
pandu_filtered = pandu_raw[pandu_raw['is_valid']].copy()
bhomo_filtered = bhomo_raw[bhomo_raw['is_valid']].copy()

print(f'\nPandu after QC   : {len(pandu_filtered)} records')
print(f'Bhomo after QC   : {len(pandu_filtered)} records')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

axes[0, 0].plot(pandu_filtered['dataTime'], pandu_filtered['dataValue'],
                linewidth=0.5, color='blue', alpha=0.7)
axes[0, 0].set_title('Pandu — Full Record')
axes[0, 0].set_ylabel('Q (m³/s)')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(bhomo_filtered['dataTime'], bhomo_filtered['dataValue'],
                linewidth=0.5, color='green', alpha=0.7)
axes[0, 1].set_title('Bhomoraguri — Full Record')
axes[0, 1].set_ylabel('Q (m³/s)')
axes[0, 1].grid(True, alpha=0.3)

pandu_study_plot = pandu_filtered[pandu_filtered['year'].between(2018, 2025)]
axes[1, 0].plot(pandu_study_plot['dataTime'], pandu_study_plot['dataValue'],
                linewidth=0.8, color='blue', alpha=0.8)
axes[1, 0].set_title('Pandu — Study Period 2018-2025')
axes[1, 0].set_ylabel('Q (m³/s)')
axes[1, 0].grid(True, alpha=0.3)

bhomo_study_plot = bhomo_filtered[bhomo_filtered['year'].between(2018, 2025)]
axes[1, 1].plot(bhomo_study_plot['dataTime'], bhomo_study_plot['dataValue'],
                linewidth=0.8, color='green', alpha=0.8)
axes[1, 1].set_title('Bhomoraguri — Study Period 2018-2025')
axes[1, 1].set_ylabel('Q (m³/s)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/01_raw_discharge_QC.png', dpi=150, bbox_inches='tight')
plt.show()


#  Restrict to study period and resample to daily
# Study period: 2018-2025


In [ ]:
study_start = '2018-01-01'
study_end   = '2025-12-31'

pandu_study = pandu_filtered[pandu_filtered['dataTime'].between(study_start, study_end)]
bhomo_study = bhomo_filtered[bhomo_filtered['dataTime'].between(study_start, study_end)]

pandu_daily = (pandu_study.set_index('dataTime')['dataValue']
               .resample('D').mean().rename('Q_pandu'))
bhomo_daily = (bhomo_study.set_index('dataTime')['dataValue']
               .resample('D').mean().rename('Q_bhomo'))

df = pd.DataFrame({'Q_pandu': pandu_daily, 'Q_bhomo': bhomo_daily})
df.index.name = 'date'
df['Year']  = df.index.year
df['Month'] = df.index.month
df['Doy']   = df.index.dayofyear

print(f'\nDaily discharge dataframe: {len(df)} days')
print(f'Date range: {df.index.min()} to {df.index.max()}')
print(f'\nMissing values:')
print(f'  Pandu NaN : {df["Q_pandu"].isna().sum()} days')
print(f'  Bhomo NaN : {df["Q_bhomo"].isna().sum()} days')


# Gaps Identified and has to be inetrpolated before analsysis.
# Analyse gap structure before interpolating

In [ ]:
def analyse_gaps(series, name):
    """Identify consecutive gap runs and their timing."""
    is_nan    = series.isna()
    gap_runs  = []
    in_gap    = False
    gap_start = None

    for date, nan in is_nan.items():
        if nan and not in_gap:
            in_gap    = True
            gap_start = date
        elif not nan and in_gap:
            gap_runs.append({
                'start'  : gap_start,
                'end'    : date,
                'length' : (date - gap_start).days,
                'month'  : gap_start.month,
                'season' : 'monsoon' if gap_start.month in [6, 7, 8, 9, 10] else 'lean'
            })
            in_gap = False

    df_gaps = pd.DataFrame(gap_runs)
    print(f'\n{name} gap analysis:')
    print(f'  Total NaN days : {is_nan.sum()}')
    print(f'  Number of gaps : {len(df_gaps)}')
    if len(df_gaps) > 0:
        print(f'  Max gap length : {df_gaps["length"].max()} days')
        print(f'  Monsoon gaps   : {len(df_gaps[df_gaps["season"] == "monsoon"])}')
        print(f'  Lean gaps      : {len(df_gaps[df_gaps["season"] == "lean"])}')
        print(f'\n  Gap details:')
        print(df_gaps.to_string(index=False))
    return df_gaps


pandu_gaps = analyse_gaps(df['Q_pandu'], 'Pandu')
bhomo_gaps = analyse_gaps(df['Q_bhomo'], 'Bhomoraguri')

Gap Assessment :

Pandu — both gaps are 1 day only. Trivial to interpolate. No concerns.

Bhomoraguri — three categories:

Category 1 — Single day gaps (17 gaps, 1-2 days): Simple linear interpolation is fine for all of these. They are instrument read failures, not genuine data gaps.

Category 2 — 28-day gap (Feb 1 to Mar 1, 2019): This is lean season so it does not affect monsoon flood metrics. However it falls exactly in  lean season composite window (Feb-Mar 2019). Since the surrounding January and March values are available, linear interpolation over 28 days is reasonable for lean season when discharge changes slowly. Flag it in your methods.

Category 3 — 26-day gap (Mar 25 to Apr 20, 2020): This is pre-monsoon transition period — not monsoon season, not lean season. Interpolation acceptable.

The 6 and 8monsoon gaps in Bhomoraguri 2018 & 2019 are all 1-day gaps in September-October — every one is interpolatable without concern.
Critical conclusion — no gap affects  monsoon peak discharge computation for any year. The longest gaps are both in lean season or pre-monsoon.Flood metrics are safe.

In [ ]:
df['pandu_interpolated'] = df['Q_pandu'].isna()
df['bhomo_interpolated'] = df['Q_bhomo'].isna()

df['Q_pandu'] = df['Q_pandu'].interpolate(method='linear', limit_area='inside')
df['Q_bhomo'] = df['Q_bhomo'].interpolate(method='linear')

# Long gap flags for transparency in methods section
df['bhomo_long_gap'] = False
df.loc['2019-02-01':'2019-03-01', 'bhomo_long_gap'] = True   # lean season
df.loc['2020-03-25':'2020-04-20', 'bhomo_long_gap'] = True   # pre-monsoon

print('\nInterpolation complete.')
print(f'Remaining Pandu NaN : {df["Q_pandu"].isna().sum()}')
print(f'Remaining Bhomo NaN : {df["Q_bhomo"].isna().sum()}')
print(f'Pandu days interpolated   : {df["pandu_interpolated"].sum()}')
print(f'Bhomo days interpolated   : {df["bhomo_interpolated"].sum()}')
print(f'Bhomo long-gap days flagged: {df["bhomo_long_gap"].sum()}')


# Section 2: Normalisation
# Compute Q/Q̄ using long-term mean from full 2001-2025 record

In [ ]:


Q_mean_pandu = pandu_filtered['dataValue'].mean()
Q_mean_bhomo = bhomo_filtered['dataValue'].mean()

print(f'Long-term mean Q (2001-2025):')
print(f'  Pandu       : {Q_mean_pandu:,.0f} m³/s')
print(f'  Bhomoraguri : {Q_mean_bhomo:,.0f} m³/s')
print(f'  Ratio P/B   : {Q_mean_pandu/Q_mean_bhomo:.4f}  (>1 expected — Pandu drains larger area)')
print(f'\n  NOTE: Do not publish raw Q values — report only Q/Q̄ ratios.')

df['Q_pandu_norm'] = df['Q_pandu'] / Q_mean_pandu
df['Q_bhomo_norm'] = df['Q_bhomo'] / Q_mean_bhomo

print(f'\nNormalised Q range (study period):')
print(f'  Pandu : {df["Q_pandu_norm"].min():.3f} to {df["Q_pandu_norm"].max():.3f}')
print(f'  Bhomo : {df["Q_bhomo_norm"].min():.3f} to {df["Q_bhomo_norm"].max():.3f}')

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
fig.suptitle('Normalised Discharge Q/Q̄ — Study Period 2018-2025',
             fontsize=12, fontweight='bold')

axes[0].plot(df.index, df['Q_pandu_norm'], linewidth=0.8, color='#2196F3', alpha=0.9)
axes[0].axhline(y=1.0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Q̄')
axes[0].set_ylabel('Q/Q̄  Pandu', fontsize=10)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(df.index, df['Q_bhomo_norm'], linewidth=0.8, color='#4CAF50', alpha=0.9)
axes[1].axhline(y=1.0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Q̄')
axes[1].set_ylabel('Q/Q̄  Bhomoraguri', fontsize=10)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/02_normalised_discharge.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
wl_raw = pd.read_csv(WL_PATH)
wl_raw['dataTime'] = pd.to_datetime(wl_raw['dataTime'])
wl_raw['year']  = wl_raw['dataTime'].dt.year
wl_raw['month'] = wl_raw['dataTime'].dt.month
wl_raw['date']  = wl_raw['dataTime'].dt.normalize()

pandu_wl_MSL  = wl_raw[(wl_raw['stationName'] == 'Pandu') & (wl_raw['dataTypeCode'] == 'HHS')].copy()
pandu_wl_zero = wl_raw[(wl_raw['stationName'] == 'Pandu') & (wl_raw['dataTypeCode'] == 'HZS')].copy()
bhomo_wl      = wl_raw[wl_raw['stationName'] == 'Bhomoraguri'].copy()

pandu_wl_daily = (pandu_wl_MSL.set_index('dataTime')['dataValue']
                  .resample('D').mean().rename('WL_pandu'))
bhomo_wl_daily = (bhomo_wl.set_index('dataTime')['dataValue']
                  .resample('D').mean().rename('WL_bhomo'))

rating_pandu = pd.DataFrame({'Q': df['Q_pandu'], 'WL': pandu_wl_daily}).dropna()
rating_bhomo = pd.DataFrame({'Q': df['Q_bhomo'], 'WL': bhomo_wl_daily}).dropna()

print(f'\nPandu rating curve pairs : {len(rating_pandu)}')
print(f'Bhomo rating curve pairs : {len(rating_bhomo)}')

In [ ]:
def power_law_rating(h, a, b, e):
    """Standard CWC rating curve: Q = a × (h − e)^b."""
    return a * np.power(np.maximum(h - e, 0.001), b)


h_pandu = rating_pandu['WL'].values
q_pandu = rating_pandu['Q'].values
e0_pandu = rating_pandu['WL'].min() - 0.5

try:
    popt_pandu, _ = curve_fit(
        power_law_rating, h_pandu, q_pandu,
        p0=([100, 2.0, e0_pandu]),
        bounds=([0, 0.5, 30], [1e6, 5.0, 45]),
        maxfev=10000
    )
    a_p, b_p, e_p = popt_pandu
    q_fit_pandu   = power_law_rating(h_pandu, *popt_pandu)
    r2_pandu      = pearsonr(q_pandu, q_fit_pandu)[0] ** 2
    print(f'\nPandu rating curve : Q = {a_p:.2f} × (WL − {e_p:.2f})^{b_p:.3f}  R² = {r2_pandu:.4f}')
    Q_bkf_pandu   = power_law_rating(48.68, *popt_pandu)
    print(f'  Bankfull at warning level (48.68 m): {Q_bkf_pandu:,.0f} m³/s  →  Q/Q̄ = {Q_bkf_pandu/Q_mean_pandu:.3f}')
except Exception as ex:
    print(f'Pandu curve fit failed: {ex}')
    Q_bkf_pandu = Q_mean_pandu * 1.40   # fallback: 1.40 × mean

h_bhomo = rating_bhomo['WL'].values
q_bhomo = rating_bhomo['Q'].values
e0_bhomo = rating_bhomo['WL'].min() - 0.5

try:
    popt_bhomo, _ = curve_fit(
        power_law_rating, h_bhomo, q_bhomo,
        p0=([100, 2.0, e0_bhomo]),
        bounds=([0, 0.5, 50], [1e6, 5.0, 62]),
        maxfev=10000
    )
    a_b, b_b, e_b = popt_bhomo
    q_fit_bhomo   = power_law_rating(h_bhomo, *popt_bhomo)
    r2_bhomo      = pearsonr(q_bhomo, q_fit_bhomo)[0] ** 2
    print(f'\nBhomo  rating curve : Q = {a_b:.2f} × (WL − {e_b:.2f})^{b_b:.3f}  R² = {r2_bhomo:.4f}')
    Q_bkf_bhomo   = power_law_rating(64.23, *popt_bhomo)
    print(f'  Bankfull at warning level (64.23 m): {Q_bkf_bhomo:,.0f} m³/s  →  Q/Q̄ = {Q_bkf_bhomo/Q_mean_bhomo:.3f}')
except Exception as ex:
    print(f'Bhomo curve fit failed: {ex}')
    Q_bkf_bhomo = Q_mean_bhomo * 1.65   # fallback

In [ ]:
def wolman_miller_bankfull(daily_q, k=1.7, n_bins=150):
    """Dominant discharge via effectiveness = Q^k × f(Q)."""
    q_valid    = daily_q.dropna().values
    counts, edges = np.histogram(q_valid, bins=n_bins, density=True)
    q_bins     = (edges[:-1] + edges[1:]) / 2
    eff_smooth = gaussian_filter1d((q_bins ** k) * counts, sigma=3)
    Q_dom      = q_bins[np.argmax(eff_smooth)]
    return Q_dom, q_bins, eff_smooth, counts


def fit_segmented_rating(h, q):
    """Piecewise two-segment power law; breakpoint = bankfull stage."""
    h_lo, h_hi   = np.percentile(h, 25), np.percentile(h, 80)
    h_candidates = np.linspace(h_lo, h_hi, 60)

    def two_seg_sse(h_break):
        m1, m2 = h <= h_break, h > h_break
        if m1.sum() < 5 or m2.sum() < 5:
            return 1e18
        try:
            p1, _ = curve_fit(power_law_rating, h[m1], q[m1],
                               p0=[50, 2, h[m1].min() - 0.3],
                               bounds=([0, 0.3, h.min() - 5], [1e7, 6, h_break]),
                               maxfev=5000)
            p2, _ = curve_fit(power_law_rating, h[m2], q[m2],
                               p0=[50, 2, h[m2].min() - 0.3],
                               bounds=([0, 0.3, h_break - 5], [1e7, 6, h.max()]),
                               maxfev=5000)
            return (np.sum((q[m1] - power_law_rating(h[m1], *p1)) ** 2) +
                    np.sum((q[m2] - power_law_rating(h[m2], *p2)) ** 2))
        except Exception:
            return 1e18

    sse_vals = [two_seg_sse(hb) for hb in h_candidates]
    h_break  = h_candidates[np.argmin(sse_vals)]
    m1, m2   = h <= h_break, h > h_break
    p1, _ = curve_fit(power_law_rating, h[m1], q[m1],
                      p0=[50, 2, h[m1].min() - 0.3],
                      bounds=([0, 0.3, h.min() - 5], [1e7, 6, h_break]), maxfev=5000)
    p2, _ = curve_fit(power_law_rating, h[m2], q[m2],
                      p0=[50, 2, h[m2].min() - 0.3],
                      bounds=([0, 0.3, h_break - 5], [1e7, 6, h.max()]), maxfev=5000)
    Q_break = power_law_rating(h_break, *p1)
    return h_break, Q_break, p1, p2


Q_wm_pandu, qb_p, eff_p, cnt_p = wolman_miller_bankfull(df['Q_pandu'])
Q_wm_bhomo, qb_b, eff_b, cnt_b = wolman_miller_bankfull(df['Q_bhomo'])

h_break_pandu, Q_bkf_seg_pandu, p1_pandu, p2_pandu = fit_segmented_rating(h_pandu, q_pandu)
h_break_bhomo, Q_bkf_seg_bhomo, p1_bhomo, p2_bhomo = fit_segmented_rating(h_bhomo, q_bhomo)

print(f'\nPandu   — W-M dominant Q : {Q_wm_pandu:,.0f} m³/s  (Q/Q̄ = {Q_wm_pandu/Q_mean_pandu:.3f})')
print(f'Pandu   — Seg. rating h* : {h_break_pandu:.2f} m  →  Q = {Q_bkf_seg_pandu:,.0f} m³/s  (Q/Q̄ = {Q_bkf_seg_pandu/Q_mean_pandu:.3f})')
print(f'Bhomo   — W-M dominant Q : {Q_wm_bhomo:,.0f} m³/s  (Q/Q̄ = {Q_wm_bhomo/Q_mean_bhomo:.3f})')
print(f'Bhomo   — Seg. rating h* : {h_break_bhomo:.2f} m  →  Q = {Q_bkf_seg_bhomo:,.0f} m³/s  (Q/Q̄ = {Q_bkf_seg_bhomo/Q_mean_bhomo:.3f})')

In [ ]:
# Use segmented rating bankfull as the primary threshold (most physically grounded)
Q_bkf_pandu_norm = Q_bkf_seg_pandu / Q_mean_pandu
Q_bkf_bhomo_norm = Q_bkf_seg_bhomo / Q_mean_bhomo

print(f'\nFinal bankfull thresholds used in flood metric computation:')
print(f'  Bhomoraguri Q_bkf/Q̄ : {Q_bkf_bhomo_norm:.3f}')
print(f'  Pandu       Q_bkf/Q̄ : {Q_bkf_pandu_norm:.3f}')


In [ ]:
# ── Rating curve figure ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    'Geomorphic Bankfull Discharge — Brahmaputra at Pandu & Bhomoraguri\n'
    'Segmented Rating Curve (top)  |  Wolman-Miller Effectiveness (bottom)',
    fontsize=12, fontweight='bold'
)

for col, (h, q, q_fit, popt, h_break, Q_bkf, Q_wm,
          q_bins, eff_smooth, counts,
          p1, p2, station, Q_mean, color) in enumerate([
    (h_pandu, q_pandu, q_fit_pandu, popt_pandu,
     h_break_pandu, Q_bkf_seg_pandu, Q_wm_pandu,
     qb_p, eff_p, cnt_p, p1_pandu, p2_pandu,
     'Pandu', Q_mean_pandu, '#2196F3'),
    (h_bhomo, q_bhomo, q_fit_bhomo, popt_bhomo,
     h_break_bhomo, Q_bkf_seg_bhomo, Q_wm_bhomo,
     qb_b, eff_b, cnt_b, p1_bhomo, p2_bhomo,
     'Bhomoraguri', Q_mean_bhomo, '#4CAF50'),
]):
    ax = axes[0, col]
    ax.scatter(h, q / Q_mean, s=6, alpha=0.2, color=color, label='Observed')
    h_range = np.linspace(h.min(), h.max() + 0.3, 300)
    ax.plot(h_range, power_law_rating(h_range, *popt) / Q_mean,
            color='grey', lw=1.2, ls='--',
            label=f'Single power law (R²={pearsonr(q, q_fit)[0]**2:.3f})')
    h1 = np.linspace(h.min(), h_break, 150)
    h2 = np.linspace(h_break, h.max() + 0.3, 150)
    ax.plot(h1, power_law_rating(h1, *p1) / Q_mean, color='navy', lw=2.2, label='Segment 1 — in-channel')
    ax.plot(h2, power_law_rating(h2, *p2) / Q_mean, color='darkorange', lw=2.2, label='Segment 2 — overbank')
    ax.axvline(h_break, color='red', ls='--', lw=1.5, label=f'Bankfull h* = {h_break:.2f} m')
    ax.axhline(Q_bkf / Q_mean, color='red', ls=':', lw=1.5, label=f'Bankfull Q/Q̄ = {Q_bkf/Q_mean:.2f}')
    ax.set_xlabel('Water Level (m MSL)', fontsize=10)
    ax.set_ylabel('Q / Q̄', fontsize=10)
    ax.set_title(f'{station} — Segmented Rating Curve', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    ax = axes[1, col]
    ax2 = ax.twinx()
    ax2.fill_between(q_bins / Q_mean, counts, alpha=0.15, color=color)
    ax2.set_ylabel('Frequency density', fontsize=9, color='grey')
    ax2.tick_params(axis='y', colors='grey')
    eff_norm = eff_smooth / eff_smooth.max()
    ax.plot(q_bins / Q_mean, eff_norm, color=color, lw=2.2, label='Q^1.7 × f(Q)')
    ax.axvline(Q_wm / Q_mean, color='red', ls='--', lw=1.5, label=f'Dominant Q/Q̄ = {Q_wm/Q_mean:.2f}')
    ax.axvline(Q_bkf / Q_mean, color='purple', ls=':', lw=1.5, label=f'Seg. rating Q/Q̄ = {Q_bkf/Q_mean:.2f}')
    ax.set_xlabel('Q / Q̄', fontsize=10)
    ax.set_ylabel('Normalised Effectiveness', fontsize=10)
    ax.set_title(f'{station} — Wolman-Miller Dominant Discharge', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/03_rating_curves_bankfull.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:


FLOOD_MONTHS      = [5, 6, 7, 8, 9, 10]
STUDY_YEARS       = list(range(2018, 2026))
MAX_PEAK_LAG_DAYS = 30    # threshold to trigger secondary peak search
PEAK_SEARCH_WIN   = 15    # ±days around Bhomo peak to search for Pandu co-event peak
MIN_COVERAGE      = 0.80  # ≥80% non-interpolated days in flood season required


In [ ]:
def count_pulses(series: pd.Series, threshold: float) -> int:
    """Count distinct periods where Q/Q̄ continuously exceeds threshold."""
    above    = (series >= threshold).values
    in_pulse = False
    pulses   = 0
    for val in above:
        if val and not in_pulse:
            in_pulse = True
            pulses  += 1
        elif not val:
            in_pulse = False
    return pulses


def compute_travel_time(bhomo_series: pd.Series,
                        pandu_series: pd.Series,
                        bkf_bhomo: float,
                        bkf_pandu: float,
                        max_lag: int = 10) -> int:
    """
    Cross-correlation travel time (days), restricted to active-flood days only.
    Positive lag = Pandu lags Bhomo (physically correct).
    """
    active = (bhomo_series >= bkf_bhomo) | (pandu_series >= bkf_pandu)
    if active.sum() < 10:
        return np.nan

    b = bhomo_series[active].values.astype(float)
    p = pandu_series[active].values.astype(float)
    b = (b - b.mean()) / (b.std() + 1e-9)
    p = (p - p.mean()) / (p.std() + 1e-9)

    corr  = correlate(p, b, mode='full')
    lags  = np.arange(-(len(b) - 1), len(b))
    valid = (lags >= 0) & (lags <= max_lag)
    return int(lags[valid][np.argmax(corr[valid])])


def season_coverage(year: int, interp_flag_col: str, dataframe: pd.DataFrame) -> float:
    """Fraction of flood-season days that are original (not interpolated)."""
    mask   = (dataframe['Year'] == year) & (dataframe['Month'].isin(FLOOD_MONTHS))
    subset = dataframe[mask]
    if len(subset) == 0:
        return 0.0
    original = (~subset[interp_flag_col]).sum()
    return original / len(subset)


In [ ]:
flood_metrics = []

for year in STUDY_YEARS:

    mask = (df['Year'] == year) & (df['Month'].isin(FLOOD_MONTHS))
    season_b = df[mask]['Q_bhomo_norm'].dropna()
    season_p = df[mask]['Q_pandu_norm'].dropna()

    if len(season_b) == 0 or len(season_p) == 0:
        continue

    cov_bhomo = season_coverage(year, 'bhomo_interpolated', df)
    cov_pandu = season_coverage(year, 'pandu_interpolated', df)
    data_quality_flag = ''

    if cov_bhomo < MIN_COVERAGE or cov_pandu < MIN_COVERAGE:
        data_quality_flag = f'LOW_COVERAGE-Bhomo:{cov_bhomo:.0%}_Pandu:{cov_pandu:.0%}'
        print(f'  [WARNING] {year}: {data_quality_flag}')

    # [B4] Flag 2018 for uncertain Pandu peak
    if year == 2018:
        data_quality_flag = (data_quality_flag + '|PANDU_DATA_UNCERTAIN').lstrip('|')

    # [B3] Flag 2025 for October data absence
    if year == 2025:
        oct_nan = df[(df['Year'] == 2025) & (df['Month'] == 10)]['Q_pandu'].isna().sum()
        if oct_nan > 25:
            data_quality_flag = (data_quality_flag + '|PANDU_OCT_GAP').lstrip('|')

    peak_bhomo      = season_b.max()
    peak_bhomo_date = season_b.idxmax()

    # Global annual Pandu peak (true maximum regardless of event)
    peak_pandu_global      = season_p.max()
    peak_pandu_global_date = season_p.idxmax()

    # [B2] Co-event Pandu peak: local maximum within ±PEAK_SEARCH_WIN days of Bhomo peak
    w0 = peak_bhomo_date - pd.Timedelta(days=PEAK_SEARCH_WIN)
    w1 = peak_bhomo_date + pd.Timedelta(days=PEAK_SEARCH_WIN)
    local_window = season_p[(season_p.index >= w0) & (season_p.index <= w1)]

    if len(local_window) > 0:
        peak_pandu_coevent      = local_window.max()
        peak_pandu_coevent_date = local_window.idxmax()
    else:
        peak_pandu_coevent      = peak_pandu_global
        peak_pandu_coevent_date = peak_pandu_global_date

    lag_global = abs((peak_pandu_global_date - peak_bhomo_date).days)
    same_event = lag_global <= MAX_PEAK_LAG_DAYS

    if not same_event and not data_quality_flag.endswith('DIFF_EVENT'):
        data_quality_flag = (data_quality_flag + '|DIFF_EVENT').lstrip('|')
        print(f'  [INFO] {year}: Pandu global peak ({peak_pandu_global_date.strftime("%b-%d")}) '
              f'is {lag_global}d from Bhomo peak — different events. '
              f'Co-event peak used for attenuation ratio.')

    # attenuation_ratio uses co-event peaks (same flood wave)
    attenuation = (peak_bhomo - peak_pandu_coevent) / peak_bhomo

    # ── Duration above bankfull ───────────────────────────────────────────────
    dur_bhomo = int((season_b >= Q_bkf_bhomo_norm).sum())
    dur_pandu = int((season_p >= Q_bkf_pandu_norm).sum())

    # ── Cumulative excess discharge ───────────────────────────────────────────
    excess_bhomo = float((season_b - Q_bkf_bhomo_norm).clip(lower=0).sum())
    excess_pandu = float((season_p - Q_bkf_pandu_norm).clip(lower=0).sum())

    # ── Flood pulses ──────────────────────────────────────────────────────────
    pulses_bhomo = count_pulses(season_b, Q_bkf_bhomo_norm)
    pulses_pandu = count_pulses(season_p, Q_bkf_pandu_norm)

    # ── Peak timing ───────────────────────────────────────────────────────────
    peak_doy_bhomo = peak_bhomo_date.dayofyear
    peak_doy_pandu = peak_pandu_global_date.dayofyear   # report global peak DOY

    # ── Travel time ───────────────────────────────────────────────────────────
    travel_days = compute_travel_time(
        season_b, season_p, Q_bkf_bhomo_norm, Q_bkf_pandu_norm
    )

    flood_metrics.append({
        'year'                  : year,
        'peak_bhomo'            : round(peak_bhomo, 3),
        'peak_pandu_global'     : round(peak_pandu_global, 3),      # [B2] NEW
        'peak_pandu_coevent'    : round(peak_pandu_coevent, 3),     # [B2] NEW
        'peak_bhomo_date'       : peak_bhomo_date.strftime('%Y-%m-%d'),
        'peak_pandu_global_date': peak_pandu_global_date.strftime('%Y-%m-%d'),
        'peak_pandu_coevent_date': peak_pandu_coevent_date.strftime('%Y-%m-%d'),
        'same_event'            : int(same_event),
        'peak_doy_bhomo'        : peak_doy_bhomo,
        'peak_doy_pandu'        : peak_doy_pandu,
        'dur_bhomo'             : dur_bhomo,
        'dur_pandu'             : dur_pandu,
        'excess_bhomo'          : round(excess_bhomo, 3),
        'excess_pandu'          : round(excess_pandu, 3),
        'pulses_bhomo'          : pulses_bhomo,
        'pulses_pandu'          : pulses_pandu,
        'travel_days'           : travel_days,
        'attenuation_ratio'     : round(attenuation, 3),
        'cov_bhomo_pct'         : round(cov_bhomo * 100, 1),
        'cov_pandu_pct'         : round(cov_pandu * 100, 1),
        'data_quality'          : data_quality_flag if data_quality_flag else 'OK',
    })



df_flood = pd.DataFrame(flood_metrics).set_index('year')

print('\n' + '=' * 72)
print('ANNUAL FLOOD METRICS — May–October flood season')
print(f'Bankfull Bhomoraguri Q/Q̄ : {Q_bkf_bhomo_norm:.3f}')
print(f'Bankfull Pandu       Q/Q̄ : {Q_bkf_pandu_norm:.3f}')
print(f'Normalisation base        : long-term mean Q (2001–2025)')
print(f'Min coverage required     : {MIN_COVERAGE:.0%}')
print('=' * 72)

display_cols = [
    'peak_bhomo', 'peak_pandu_global', 'peak_pandu_coevent',
    'peak_bhomo_date', 'peak_pandu_global_date',
    'peak_doy_bhomo', 'peak_doy_pandu',
    'dur_bhomo', 'dur_pandu',
    'excess_bhomo', 'excess_pandu',
    'pulses_bhomo', 'pulses_pandu',
    'travel_days', 'attenuation_ratio', 'data_quality'
]
print('\n--- ALL YEARS (with quality flags) ---')
print(df_flood[display_cols].to_string())

df_flood_clean   = df_flood[df_flood['data_quality'] == 'OK'].copy()
df_flood_flagged = df_flood[df_flood['data_quality'] != 'OK'].copy()

if len(df_flood_flagged) > 0:
    print(f'\n--- FLAGGED YEARS ---')
    print(df_flood_flagged[['data_quality', 'cov_bhomo_pct', 'cov_pandu_pct',
                             'peak_bhomo_date', 'peak_pandu_global_date',
                             'same_event']].to_string())



In [ ]:
EHE_EXPONENT = 1.7   # Wolman-Miller (1960) — sediment transport scaling
TYPICAL_PEAK_DOY = 196  # long-term median peak day (Jul 15)

new_index_rows = []

for year in STUDY_YEARS:
    mask     = (df['Year'] == year) & (df['Month'].isin(FLOOD_MONTHS))
    season_b = df[mask]['Q_bhomo_norm'].dropna()
    season_p = df[mask]['Q_pandu_norm'].dropna()

    if len(season_b) == 0 or len(season_p) == 0:
        continue

    # Retrieve values already computed in Section 3
    exc_b = df_flood.loc[year, 'excess_bhomo']
    exc_p = df_flood.loc[year, 'excess_pandu']
    dur_b = df_flood.loc[year, 'dur_bhomo']
    dur_p = df_flood.loc[year, 'dur_pandu']

    # ── RIFW — Reach Internal Flood Work ─────────────────────────────────────
    # Daily difference (Q_bhomo − Q_pandu) on days when Bhomo is above bankfull.
    # Summing these differences gives the net energy retained or amplified within reach.
    df_season = df[mask].copy()
    daily_diff = (
        df_season['Q_bhomo_norm'] - df_season['Q_pandu_norm']
    ).where(df_season['Q_bhomo_norm'] >= Q_bkf_bhomo_norm, other=0).fillna(0)
    RIFW = float(daily_diff.sum())

    # ── UDR — Upstream Dominance Ratio ───────────────────────────────────────
    UDR = dur_b / dur_p if dur_p > 0 else np.nan

    # ── IRTC — Intra-reach Tributary Contribution ─────────────────────────────
    IRTC = (exc_p - exc_b) / exc_b if exc_b > 0 else np.nan

    # ── EHE — Effective Hydraulic Energy (Wolman-Miller) ─────────────────────
    # Apply separately to each station using that station's bankfull threshold.
    # This preserves the physical interpretation: EHE_bhomo is the geomorphic
    # energy of the upstream boundary condition; EHE_pandu is the integrated
    # reach output energy including tributary contributions.
    above_bkf_b = season_b[season_b >= Q_bkf_bhomo_norm]
    above_bkf_p = season_p[season_p >= Q_bkf_pandu_norm]
    EHE_bhomo = float((above_bkf_b ** EHE_EXPONENT).sum())
    EHE_pandu = float((above_bkf_p ** EHE_EXPONENT).sum())

    # ── PDD — Peak Day-of-Year Deviation ─────────────────────────────────────
    peak_doy = df_flood.loc[year, 'peak_doy_bhomo']
    PDD = int(peak_doy - TYPICAL_PEAK_DOY)

    # ── FPI — Flood Persistence Index ────────────────────────────────────────
    n_season_days = len(season_b)
    FPI_bhomo = dur_b / n_season_days if n_season_days > 0 else np.nan
    FPI_pandu = dur_p / len(season_p)  if len(season_p) > 0 else np.nan

    new_index_rows.append({
        'year'     : year,
        'RIFW'     : round(RIFW,      3),
        'UDR'      : round(UDR,       3) if not np.isnan(UDR)  else np.nan,
        'IRTC'     : round(IRTC,      3) if not np.isnan(IRTC) else np.nan,
        'EHE_bhomo': round(EHE_bhomo, 2),
        'EHE_pandu': round(EHE_pandu, 2),
        'PDD'      : PDD,
        'FPI_bhomo': round(FPI_bhomo, 3) if not np.isnan(FPI_bhomo) else np.nan,
        'FPI_pandu': round(FPI_pandu, 3) if not np.isnan(FPI_pandu) else np.nan,
    })

df_new = pd.DataFrame(new_index_rows).set_index('year')

print('\n--- NEW INDICES (2018–2025) ---')
print(df_new.to_string())

print('\nPhysical interpretation notes:')
print('  RIFW  > 0 : reach absorbs flood energy (attenuation regime)')
print('  RIFW  < 0 : tributary surplus exceeds upstream input (amplification)')
print('  UDR   < 1 : tributaries sustain Pandu flooding > Bhomo duration (all years here)')
print('  IRTC  > 0 : tributaries add flood volume (always positive in this dataset)')
print('  EHE       : higher = more geomorphically effective flooding (disproportionate bar work)')
print('  PDD   > 0 : late-season flood (slower post-flood vegetation recovery)')
print('  FPI       : fraction of monsoon season above bankfull (0–1 scale)')


# ── Merge all metrics into single enhanced dataframe ─────────────────────────
df_flood_enhanced = df_flood.join(df_new, how='left')

print('\n--- ENHANCED FLOOD METRICS — ALL COLUMNS ---')
print(df_flood_enhanced.to_string())

# Export to CSV
out_path = f'{FIGURES_DIR}/flood_metrics_enhanced.csv'
df_flood_enhanced.to_csv(out_path)
print(f'\nExported: {out_path}')


In [ ]:
years = df_flood_enhanced.index.values
x     = np.arange(len(years))

fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
fig.suptitle(
    'Flood-Derived Morphodynamic Metrics — Brahmaputra Reach\n'
    'RIFW | UDR | IRTC | EHE | FPI | PDD',
    fontsize=12, fontweight='bold'
)

# ── Panel definitions ─────────────────────────────
PANELS = [
    ('RIFW', None, 'RIFW (Q/Q̄·days)', '#2E7D32'),
    ('UDR', None, 'UDR (–)', '#1565C0'),
    ('IRTC', None, 'IRTC (–)', '#7B1FA2'),
    ('EHE_bhomo', 'EHE_pandu', 'EHE (Bhomo vs Pandu)', None),
    ('FPI_bhomo', 'FPI_pandu', 'FPI (Bhomo vs Pandu)', None),
    ('PDD', None, 'PDD (days)', '#FF6F00'),
]

for idx, (col1, col2, ylabel, color) in enumerate(PANELS):
    row = idx // 2
    col_i = idx % 2
    ax = axes[row, col_i]

    y1 = df_flood_enhanced[col1].astype(float).values

    # ── Single-variable panels ─────────────────────
    if col2 is None:
        ax.plot(x, y1,
                color=color, lw=2,
                marker='o', ms=5)

        # Zero reference where needed
        if col1 in ['RIFW', 'UDR', 'IRTC']:
            ax.axhline(0, color='black', lw=0.8, ls='--')

        # Annotate values
        for xi, v in zip(x, y1):
            offset = 0.03 * (np.nanmax(y1) - np.nanmin(y1) + 1e-6)
            ax.text(xi, v + offset, f'{v:.2f}',
                    ha='center', fontsize=7)

    # ── Paired panels (Bhomo vs Pandu) ─────────────
    else:
        y2 = df_flood_enhanced[col2].astype(float).values

        ax.plot(x, y1,
                color='blue', lw=2,
                marker='o', ms=5,
                label='Bhomoraguri')

        ax.plot(x, y2,
                color='red', lw=2,
                marker='o', ms=5,
                label='Pandu')

        ax.legend(fontsize=8)

    # ── Formatting ────────────────────────────────
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(ylabel, fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

# ── X-axis only on bottom row ─────────────────────
for ax in axes[-1, :]:
    ax.set_xticks(x)
    ax.set_xticklabels(years, rotation=45)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/flood_metrics_3x2_final.png',
            dpi=300, bbox_inches='tight')

plt.show()
print('3x2 paired figure saved.')

In [ ]:
Q_bkf_pandu_norm = Q_bkf_pandu / Q_mean_pandu
Q_bkf_bhomo_norm = Q_bkf_bhomo / Q_mean_bhomo

# ── Check flood season onset ──────────────────────────────────────────────
# Look at 2020 as a representative high flood year

df_2020 = df[df['Year'] == 2020].copy()

print('2020 Bhomoraguri — monthly mean Q/Q̄:')
print(df_2020.groupby('Month')['Q_bhomo_norm'].mean().round(3))

print('\n2020 Bhomoraguri — first date exceeding bankfull:')
above_bkf = df_2020[df_2020['Q_bhomo_norm'] >= Q_bkf_bhomo_norm]
if len(above_bkf) > 0:
    print(f'  First exceedance: {above_bkf.index[0].strftime("%Y-%m-%d")}')
    print(f'  Last exceedance : {above_bkf.index[-1].strftime("%Y-%m-%d")}')
else:
    print('  No bankfull exceedance in 2020')

print('\n2020 Pandu — first date exceeding bankfull:')
above_bkf_p = df_2020[df_2020['Q_pandu_norm'] >= Q_bkf_pandu_norm]
if len(above_bkf_p) > 0:
    print(f'  First exceedance: {above_bkf_p.index[0].strftime("%Y-%m-%d")}')
    print(f'  Last exceedance : {above_bkf_p.index[-1].strftime("%Y-%m-%d")}')
else:
    print('  No bankfull exceedance in 2020')

# Section 3: Flood Metrics — Corrected

## Bugs Fixed

| # | Bug | Fix |
|---|-----|-----|
| F1 | `mask_b` and `mask_p` were identical | Merged into single `mask` |
| F2 | No coverage guard — mid-season gaps produced spurious peaks | Per-year coverage check (>=80%); flagged years excluded from trends |
| F3 | `active` mask computed but **never applied** in travel-time function | Subset `b` and `p` to active-flood days before cross-correlating |
| F4 | `idxmax()` on interpolated data gives wrong Pandu peak date | Secondary search: if Pandu peak >30d from Bhomo, find local max within +/-15d |
| F5 | `count_pulses` re-defined inside loop every iteration | Moved outside loop |

## Root Cause of 2025 Anomaly
The Sep-22 Pandu peak (DOY 265) and 9-pulse Bhomo count in 2025 are consistent with a **mid-season data gap** at Pandu (likely Jul-Aug). Linear interpolation over a long monsoon gap suppresses values, shifting the apparent peak to September. Fix F2 detects and flags this automatically.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FLOOD METRICS — CORRECTED VERSION
# Fixes applied:
#   [F1] Removed redundant/identical mask_b / mask_p
#   [F2] Data-coverage guard: years with >20% missing in flood season are flagged
#        and excluded from metrics rather than producing spurious values
#   [F3] Travel-time cross-correlation now correctly subsets the active-flood
#        period before correlating (the active mask was computed but never applied)
#   [F4] Secondary peak search: if Pandu peak is >30 days from Bhomo peak,
#        find the local maximum within ±15 days of Bhomo peak instead
#   [F5] count_pulses moved outside the loop (no re-definition overhead)
#   [F6] Explicit data-quality flags propagated to output table
# ══════════════════════════════════════════════════════════════════════════════

from scipy.signal import correlate
import numpy as np
import pandas as pd

FLOOD_MONTHS     = [5, 6, 7, 8, 9, 10]
STUDY_YEARS      = list(range(2018, 2026))
MAX_PEAK_LAG_DAYS = 30   # if Pandu peak is >30 days from Bhomo, trigger secondary search
MIN_COVERAGE     = 0.80  # require ≥80 % non-interpolated days in flood season


# ── [F5] Helper functions defined ONCE outside the loop ──────────────────────

def count_pulses(series: pd.Series, threshold: float) -> int:
    """Count distinct periods where Q/Q̄ continuously exceeds threshold."""
    above    = (series >= threshold).values
    in_pulse = False
    pulses   = 0
    for val in above:
        if val and not in_pulse:
            in_pulse = True
            pulses  += 1
        elif not val:
            in_pulse = False
    return pulses


def compute_travel_time(bhomo_series: pd.Series,
                        pandu_series: pd.Series,
                        bkf_bhomo: float,
                        bkf_pandu: float,
                        max_lag: int = 10) -> int:
    """
    [F3] FIXED — Cross-correlation is now performed only on the active-flood
    sub-period (days where at least one station exceeds bankfull).
    Positive lag = Pandu lags Bhomo (physically correct direction).
    Returns integer lag in days; np.nan if insufficient active days.
    """
    active = (bhomo_series >= bkf_bhomo) | (pandu_series >= bkf_pandu)
    if active.sum() < 10:
        return np.nan

    # [F3] Subset to active period — this was the critical missing step
    b = bhomo_series[active].values.astype(float)
    p = pandu_series[active].values.astype(float)

    # Z-score normalise before correlating
    b = (b - b.mean()) / (b.std() + 1e-9)
    p = (p - p.mean()) / (p.std() + 1e-9)

    corr = correlate(p, b, mode='full')
    lags  = np.arange(-(len(b) - 1), len(b))

    # Restrict to physically realistic travel times (0–max_lag days)
    valid     = (lags >= 0) & (lags <= max_lag)
    best_lag  = int(lags[valid][np.argmax(corr[valid])])
    return best_lag


def find_peak_with_fallback(season_series: pd.Series,
                            reference_date: pd.Timestamp,
                            max_lag_days: int = MAX_PEAK_LAG_DAYS,
                            search_window: int = 15) -> tuple:
    """
    [F4] Return (peak_value, peak_date).
    Primary   : global seasonal maximum.
    Secondary : if peak is >max_lag_days from reference_date, search within
                ±search_window days of reference_date for local maximum.
    Returns a flag indicating whether fallback was used.
    """
    peak_val  = season_series.max()
    peak_date = season_series.idxmax()
    fallback  = False

    lag_days = abs((peak_date - reference_date).days)
    if lag_days > max_lag_days:
        # Search for local max near the reference date
        window_start = reference_date - pd.Timedelta(days=search_window)
        window_end   = reference_date + pd.Timedelta(days=search_window)
        local_window = season_series[
            (season_series.index >= window_start) &
            (season_series.index <= window_end)
        ]
        if len(local_window) > 0:
            peak_val  = local_window.max()
            peak_date = local_window.idxmax()
            fallback  = True

    return peak_val, peak_date, fallback


# ── [F2] Pre-compute per-year, per-station coverage in flood season ───────────

def season_coverage(year: int,
                    interpolated_flag_col: str,
                    df: pd.DataFrame) -> float:
    """
    Fraction of flood-season days that are NOT interpolated.
    A day is 'original' if interpolated_flag_col is False.
    """
    mask   = (df['Year'] == year) & (df['Month'].isin(FLOOD_MONTHS))
    subset = df[mask]
    if len(subset) == 0:
        return 0.0
    original = (~subset[interpolated_flag_col]).sum()
    return original / len(subset)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

flood_metrics = []

for year in STUDY_YEARS:

    # [F1] Single combined mask — no more redundant mask_b / mask_p
    mask = (df['Year'] == year) & (df['Month'].isin(FLOOD_MONTHS))

    season_b = df[mask]['Q_bhomo_norm'].dropna()
    season_p = df[mask]['Q_pandu_norm'].dropna()

    if len(season_b) == 0 or len(season_p) == 0:
        continue

    # [F2] Coverage check ─────────────────────────────────────────────────────
    cov_bhomo = season_coverage(year, 'bhomo_interpolated', df)
    cov_pandu = season_coverage(year, 'pandu_interpolated', df)
    data_quality_flag = ''

    if cov_bhomo < MIN_COVERAGE or cov_pandu < MIN_COVERAGE:
        data_quality_flag = (
            f'LOW_COVERAGE — Bhomo:{cov_bhomo:.0%} Pandu:{cov_pandu:.0%}'
        )
        print(f'  [WARNING] {year}: {data_quality_flag} — metrics flagged, excluded from trends')
        # Record the flag but skip appending to metrics used in statistical analysis
        # (still append with flag so the table is complete for transparency)

    # ── Metric 1: Peak Q/Q̄ ───────────────────────────────────────────────────
    peak_bhomo      = season_b.max()
    peak_bhomo_date = season_b.idxmax()

    # [F4] Pandu peak with fallback if peak is implausibly far from Bhomo peak
    peak_pandu, peak_pandu_date, pandu_fallback = find_peak_with_fallback(
        season_p, peak_bhomo_date
    )
    if pandu_fallback:
        print(f'  [INFO] {year}: Pandu secondary peak search used '
              f'(global peak was >{MAX_PEAK_LAG_DAYS}d from Bhomo peak)')
        if not data_quality_flag:
            data_quality_flag = 'PANDU_PEAK_FALLBACK'

    # ── Metric 2: Flood duration above bankfull ───────────────────────────────
    dur_bhomo = int((season_b >= Q_bkf_bhomo_norm).sum())
    dur_pandu = int((season_p >= Q_bkf_pandu_norm).sum())

    # ── Metric 3: Cumulative excess discharge (flood volume proxy) ────────────
    excess_bhomo = float((season_b - Q_bkf_bhomo_norm).clip(lower=0).sum())
    excess_pandu = float((season_p - Q_bkf_pandu_norm).clip(lower=0).sum())

    # ── Metric 4: Number of flood pulses ─────────────────────────────────────
    pulses_bhomo = count_pulses(season_b, Q_bkf_bhomo_norm)
    pulses_pandu = count_pulses(season_p, Q_bkf_pandu_norm)

    # ── Metric 5: Flood timing — day of year of peak ──────────────────────────
    peak_doy_bhomo = peak_bhomo_date.dayofyear
    peak_doy_pandu = peak_pandu_date.dayofyear

    # ── Metric 6: Travel time (cross-correlation, active period only) ─────────
    # [F3] Fixed: active mask is now applied inside compute_travel_time
    travel_days = compute_travel_time(
        season_b, season_p,
        Q_bkf_bhomo_norm, Q_bkf_pandu_norm
    )

    # ── Metric 7: Attenuation ratio ───────────────────────────────────────────
    attenuation = (peak_bhomo - peak_pandu) / peak_bhomo

    flood_metrics.append({
        'year'              : year,
        'peak_bhomo'        : round(peak_bhomo,      3),
        'peak_pandu'        : round(peak_pandu,       3),
        'peak_bhomo_date'   : peak_bhomo_date.strftime('%Y-%m-%d'),
        'peak_pandu_date'   : peak_pandu_date.strftime('%Y-%m-%d'),
        'peak_doy_bhomo'    : peak_doy_bhomo,
        'peak_doy_pandu'    : peak_doy_pandu,
        'dur_bhomo'         : dur_bhomo,
        'dur_pandu'         : dur_pandu,
        'excess_bhomo'      : round(excess_bhomo,    3),
        'excess_pandu'      : round(excess_pandu,    3),
        'pulses_bhomo'      : pulses_bhomo,
        'pulses_pandu'      : pulses_pandu,
        'travel_days'       : travel_days,
        'attenuation_ratio' : round(attenuation,     3),
        'cov_bhomo_pct'     : round(cov_bhomo * 100, 1),
        'cov_pandu_pct'     : round(cov_pandu * 100, 1),
        'data_quality'      : data_quality_flag if data_quality_flag else 'OK',
    })


# ══════════════════════════════════════════════════════════════════════════════
# OUTPUT
# ══════════════════════════════════════════════════════════════════════════════

df_flood     = pd.DataFrame(flood_metrics).set_index('year')

# Separate clean years from flagged ones for statistical analysis
df_flood_clean   = df_flood[df_flood['data_quality'] == 'OK'].copy()
df_flood_flagged = df_flood[df_flood['data_quality'] != 'OK'].copy()

print('=' * 72)
print('ANNUAL FLOOD METRICS — May–October flood season')
print(f'Bankfull threshold  Bhomoraguri Q/Q̄ : {Q_bkf_bhomo_norm:.3f}')
print(f'Bankfull threshold  Pandu       Q/Q̄ : {Q_bkf_pandu_norm:.3f}')
print(f'Normalisation base  : long-term mean Q (2001–2025)')
print(f'Min coverage required           : {MIN_COVERAGE:.0%}')
print(f'Peak lag fallback window        : ±{MAX_PEAK_LAG_DAYS} days')
print('=' * 72)

# Display columns in logical groups
display_cols = [
    'peak_bhomo', 'peak_pandu',
    'peak_bhomo_date', 'peak_pandu_date',
    'peak_doy_bhomo', 'peak_doy_pandu',
    'dur_bhomo', 'dur_pandu',
    'excess_bhomo', 'excess_pandu',
    'pulses_bhomo', 'pulses_pandu',
    'travel_days', 'attenuation_ratio',
    ]

print('\n--- ALL YEARS (with quality flags) ---')
print(df_flood[display_cols].to_string())

print(f'\n--- CLEAN YEARS ONLY (used in statistical analysis) ---')
print(df_flood_clean[display_cols[:-3]].to_string())

if len(df_flood_flagged) > 0:
    print(f'\n--- FLAGGED YEARS (excluded from trends) ---')
    print(df_flood_flagged[['data_quality', 'cov_bhomo_pct', 'cov_pandu_pct',
                             'peak_bhomo_date', 'peak_pandu_date',
                             'pulses_bhomo', 'pulses_pandu']].to_string())
    print('\n  Action: Verify raw data for flagged years before including in paper.')
